In [0]:
%sql
CREATE TABLE lakehouse.test.transactions31 (
  card_number STRING,
  amount DOUBLE,
  timestamp TIMESTAMP
) 
USING DELTA;

-- INSERT INTO lakehouse.test.transactions31 VALUES ('1234567890123477', 400.00, '2022-01-01 10:00:00');

-- delete from lakehouse.test.transactions31 where card_number = '1234567890123477';

-- UPDATE lakehouse.test.transactions31  set amount = 500.00 where card_number = '1234567890123477'


In [0]:
%sql
ALTER TABLE lakehouse.test.transactions31 SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
from pyspark.sql.functions import window, col

# Read the delta table as a streaming source
df = spark\
.readStream\
.option("readChangeFeed", "true") \
.option("startingVersion", 2) \
.table("lakehouse.test.transactions31")

In [0]:
deleted_delta_data = df.filter("_change_type = 'delete'")  # For deleted records
updated_delta_data = df.filter("_change_type = 'update_postimage'")  # For updated records
insert_delta_data = df.filter("_change_type = 'insert'")  # For insert records

In [0]:
deleted_delta_data.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", '/Volumes/lakehouse/test/healthcare_input/checkpointing31/')\
    .toTable("lakehouse.test.transactions_deleted")

In [0]:
updated_delta_data.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", '/Volumes/lakehouse/test/healthcare_input/checkpointing32/')\
    .toTable("lakehouse.test.transactions_updated")

In [0]:
insert_delta_data.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", '/Volumes/lakehouse/test/healthcare_input/checkpointing33/')\
    .toTable("lakehouse.test.transactions_insert")